<a href="https://colab.research.google.com/github/dvinueza-ops/gdp-dashboard/blob/main/Taller_Pr%C3%A1ctico_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Paso 1:

In [1]:
import pandas as pd
from collections import Counter
import re

# Evitar que pandas recorte las descripciones largas con "..."
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

# 1. Cargar el dataset directamente desde GitHub (enlace raw)
url = "https://raw.githubusercontent.com/eugeniomorocho/ArtificialIntelligence/main/Unit%205.%20Machine%20Learning%20as%20an%20AI%20Component/5.3%20Unsupervised%20learning/Clase_2/Recursos/catalogo_productos.csv"
df = pd.read_csv(url)

print("=== 1. ESTRUCTURA DEL DATASET ===")
print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
print("Nombres de columnas:", df.columns.tolist())
print("\nValores nulos por columna:")
print(df.isnull().sum())

# Detectar automáticamente la columna de texto más larga (descripción)
cols_texto = df.select_dtypes(include=['object', 'string']).columns
col_desc = max(cols_texto, key=lambda c: df[c].astype(str).str.len().mean())
print(f"\n-> Columna de descripción detectada: '{col_desc}'")

# 2. Examinar 10 registros completos al azar
print("\n=== 2. MUESTRA ALEATORIA DE 10 PRODUCTOS ===")
muestra = df.sample(10, random_state=42)
for idx, row in muestra.iterrows():
    print(f"\n[Fila {idx}]")
    for col in df.columns:
        print(f"  {col}: {row[col]}")
    print("-" * 70)

# 3. Análisis de longitud (caracteres y palabras)
textos = df[col_desc].fillna("").astype(str)
df["n_caracteres"] = textos.str.len()
df["n_palabras"] = textos.str.split().str.len()

print("\n=== 3. ESTADÍSTICAS DE LONGITUD ===")
print(df[["n_caracteres", "n_palabras"]].describe().round(2))

# 4. Análisis de vocabulario
tokens = re.findall(r'\b\w+\b', " ".join(textos).lower())
vocabulario = set(tokens)

print("\n=== 4. VOCABULARIO ===")
print(f"Total de palabras (tokens): {len(tokens)}")
print(f"Palabras únicas (vocabulario): {len(vocabulario)}")
print("20 palabras más frecuentes:", Counter(tokens).most_common(20))

=== 1. ESTRUCTURA DEL DATASET ===
Filas: 120 | Columnas: 4
Nombres de columnas: ['producto_id', 'nombre', 'categoria', 'descripcion']

Valores nulos por columna:
producto_id    0
nombre         0
categoria      0
descripcion    0
dtype: int64

-> Columna de descripción detectada: 'descripcion'

=== 2. MUESTRA ALEATORIA DE 10 PRODUCTOS ===

[Fila 44]
  producto_id: P045
  nombre: Computacion modelo 45
  categoria: computacion
  descripcion: computador portatil laptop memoria teclado pantalla nuevo
----------------------------------------------------------------------

[Fila 47]
  producto_id: P048
  nombre: Hogar modelo 48
  categoria: hogar
  descripcion: lampara escritorio luz lectura regulable resistente
----------------------------------------------------------------------

[Fila 4]
  producto_id: P005
  nombre: Deporte modelo 5
  categoria: deporte
  descripcion: zapatillas calzado correr deporte ligero nuevo
----------------------------------------------------------------------

[

Al revisar la muestra aleatoria y las estadísticas del catálogo se observa que las descripciones son bastante cortas y homogéneas, con un promedio de 6.33 palabras (todas entre 6 y 7 palabras) y un vocabulario total de apenas 35 términos distintos en los 120 registros. En cuanto al estilo, no son redacciones comerciales extensas ni fichas técnicas detalladas, sino listas de palabras clave separadas por espacios donde se agrupan el tipo de producto, sus sinónimos principales y un adjetivo general al final (por ejemplo, *"telefono celular smartphone pantalla bateria camara compacto"* o *"zapatillas calzado correr deporte ligero nuevo"*).

Como varios productos dentro de una misma categoría comparten exactamente la misma base de palabras y solo cambian en el adjetivo final (como *compacto*, *recargable*, *nuevo* o *resistente*), es muy probable que TF-IDF encuentre coincidencias exactas o muy altas entre artículos casi idénticos, aunque tendrá poca información técnica específica (como marca, talla o capacidad) para diferenciar variantes dentro de un mismo grupo.

Paso 2:

In [2]:
import pandas as pd
import numpy as np
import re
import unicodedata
from sklearn.feature_extraction.text import TfidfVectorizer

# Respaldo por si se reinició la sesión de Colab
if 'df' not in globals():
    url = "https://raw.githubusercontent.com/eugeniomorocho/ArtificialIntelligence/main/Unit%205.%20Machine%20Learning%20as%20an%20AI%20Component/5.3%20Unsupervised%20learning/Clase_2/Recursos/catalogo_productos.csv"
    df = pd.read_csv(url)
    cols_texto = df.select_dtypes(include=['object', 'string']).columns
    col_desc = max(cols_texto, key=lambda c: df[c].astype(str).str.len().mean())

textos_crudos = df[col_desc].fillna("").astype(str)

# ==============================================================================
# A. REVISIÓN DE MINÚSCULAS, ACENTOS Y VOCABULARIO
# ==============================================================================
def quitar_acentos(texto):
    # Normaliza caracteres Unicode y elimina marcas diacríticas (tildes), conservando letras base
    texto_norm = unicodedata.normalize('NFKD', texto)
    return "".join([c for c in texto_norm if not unicodedata.combining(c)])

def limpiar_texto(texto):
    texto = texto.lower()                      # 1. Pasar a minúsculas
    texto = quitar_acentos(texto)              # 2. Eliminar acentos/tildes
    texto = re.sub(r'[^a-z0-9\s]', ' ', texto) # 3. Quitar signos de puntuación (mantener letras y números)
    texto = re.sub(r'\s+', ' ', texto).strip() # 4. Eliminar espacios múltiples
    return texto

textos_limpios = textos_crudos.apply(limpiar_texto)

# Medir cómo cambia el tamaño del vocabulario en cada etapa
vocab_crudo = set(" ".join(textos_crudos).split())
vocab_minusculas = set(" ".join(textos_crudos.str.lower()).split())
vocab_sin_acentos = set(" ".join(textos_limpios).split())

print("=== 1. IMPACTO EN EL VOCABULARIO ===")
print(f"Vocabulario original (con mayúsculas y signos): {len(vocab_crudo)} términos")
print(f"Solo minúsculas:                                {len(vocab_minusculas)} términos")
print(f"Minúsculas + sin acentos + sin puntuación:      {len(vocab_sin_acentos)} términos")

# ==============================================================================
# B. LISTA PERSONALIZADA DE PALABRAS VACÍAS (CONSERVANDO NEGACIONES)
# ==============================================================================
# Excluimos artículos, preposiciones neutras y conjunciones, PERO conservamos:
# - Negaciones: 'no', 'ni', 'nunca', 'tampoco', 'ningun', 'ninguna', 'nada'
# - Modificadores críticos de producto: 'sin', 'con', 'solo', 'mas', 'menos'
stopwords_personalizadas = [
    'el', 'la', 'los', 'las', 'un', 'una', 'unos', 'unas',
    'de', 'del', 'al', 'a', 'en', 'por', 'para',
    'y', 'e', 'o', 'u', 'que', 'se', 'su', 'sus',
    'es', 'son', 'como', 'este', 'esta', 'estos', 'estas',
    'lo', 'le', 'les', 'me', 'te', 'nos'
]

print("\n=== 2. LISTA DE PALABRAS VACÍAS UTILIZADA ===")
print(f"Total de stopwords: {len(stopwords_personalizadas)}")
print("Contenido:", stopwords_personalizadas)

# ==============================================================================
# C. COMPARACIÓN DE REPRESENTACIONES TF-IDF (BASE vs. UNIGRAMAS vs. BIGRAMAS)
# ==============================================================================
# 1. Resolución Base: configuración sencilla por defecto (solo minúsculas, unigramas)
vec_base = TfidfVectorizer(lowercase=True, ngram_range=(1, 1))
X_base = vec_base.fit_transform(textos_crudos)

# 2. Preprocesada - Solo palabras individuales (Unigramas)
vec_uni = TfidfVectorizer(ngram_range=(1, 1), stop_words=stopwords_personalizadas)
X_uni = vec_uni.fit_transform(textos_limpios)

# 3. Preprocesada - Palabras individuales + Bigramas
vec_bi = TfidfVectorizer(ngram_range=(1, 2), stop_words=stopwords_personalizadas)
X_bi = vec_bi.fit_transform(textos_limpios)

# Tabla comparativa
comparacion = pd.DataFrame({
    "Configuración": [
        "1. Base (crudo + minúsculas + unigramas)",
        "2. Limpio + Stopwords + Solo Unigramas (1,1)",
        "3. Limpio + Stopwords + Unigramas y Bigramas (1,2)"
    ],
    "Tamaño Vocabulario (Features)": [
        X_base.shape[1],
        X_uni.shape[1],
        X_bi.shape[1]
    ],
    "Forma Matriz (Docs x Terms)": [
        str(X_base.shape),
        str(X_uni.shape),
        str(X_bi.shape)
    ],
    "Dispersión (% de ceros)": [
        round((1.0 - X_base.nnz / (X_base.shape[0] * X_base.shape[1])) * 100, 2),
        round((1.0 - X_uni.nnz / (X_uni.shape[0] * X_uni.shape[1])) * 100, 2),
        round((1.0 - X_bi.nnz / (X_bi.shape[0] * X_bi.shape[1])) * 100, 2)
    ]
})

print("\n=== 3. COMPARACIÓN DE REPRESENTACIONES TF-IDF ===")
print(comparacion.to_string(index=False))

# ==============================================================================
# D. INSPECCIÓN DE BIGRAMAS Y NEGACIONES CONSERVADAS
# ==============================================================================
nombres_features = vec_bi.get_feature_names_out()
bigramas = [f for f in nombres_features if " " in f]

# Ordenar bigramas por peso TF-IDF acumulado en el catálogo
pesos_bi = np.asarray(X_bi.sum(axis=0)).ravel()
top_indices = pesos_bi.argsort()[::-1]
top_bigramas = [nombres_features[i] for i in top_indices if " " in nombres_features[i]][:15]

print("\n=== 4. EJEMPLOS DE BIGRAMAS RELEVANTES EN EL CATÁLOGO ===")
print("Top 15 bigramas por relevancia TF-IDF:", top_bigramas)

# Buscar bigramas que contengan negaciones o condiciones ('sin', 'no', 'con')
bigramas_criticos = [b for b in bigramas if b.split()[0] in {'sin', 'no', 'con', 'ni'}][:15]
print("Ejemplos de bigramas con negaciones/condiciones:", bigramas_criticos)

=== 1. IMPACTO EN EL VOCABULARIO ===
Vocabulario original (con mayúsculas y signos): 35 términos
Solo minúsculas:                                35 términos
Minúsculas + sin acentos + sin puntuación:      35 términos

=== 2. LISTA DE PALABRAS VACÍAS UTILIZADA ===
Total de stopwords: 36
Contenido: ['el', 'la', 'los', 'las', 'un', 'una', 'unos', 'unas', 'de', 'del', 'al', 'a', 'en', 'por', 'para', 'y', 'e', 'o', 'u', 'que', 'se', 'su', 'sus', 'es', 'son', 'como', 'este', 'esta', 'estos', 'estas', 'lo', 'le', 'les', 'me', 'te', 'nos']

=== 3. COMPARACIÓN DE REPRESENTACIONES TF-IDF ===
                                     Configuración  Tamaño Vocabulario (Features) Forma Matriz (Docs x Terms)  Dispersión (% de ceros)
          1. Base (crudo + minúsculas + unigramas)                             35                   (120, 35)                    82.00
      2. Limpio + Stopwords + Solo Unigramas (1,1)                             35                   (120, 35)                    82.00
3. Lim

Al aplicar la limpieza de minúsculas, acentos y signos de puntuación, el tamaño del vocabulario se mantuvo en 35 términos, lo que confirma que el conjunto de datos provisto ya venía normalizado en minúsculas y sin tildes. De igual forma, al filtrar nuestra lista personalizada de palabras vacías (artículos y preposiciones neutras como *el, la, de, en, para*), el número de rasgos no cambió porque las descripciones del catálogo están construidas directamente con palabras clave sin conectores. Aun así, se decidió definir explícitamente esa lista excluyendo negaciones (*no, ni, nunca*) y modificadores como *sin* o *con*, ya que en un catálogo real eliminar esas palabras invertiría el sentido del producto (por ejemplo, confundir un equipo *con cable* con uno *sin cable*).

Al comparar la configuración base de solo palabras individuales `(1, 1)` frente a la representación con unigramas y bigramas `(1, 2)`, la matriz pasa de 35 a 91 características y su dispersión sube de 82.00% a 87.22%. Aunque la matriz con bigramas es más grande, aporta valor porque logra capturar pares de términos que suelen ir juntos en el producto (como *auriculares bluetooth*, *luz lectura* o *zapatillas calzado*), manteniendo una referencia más precisa del contexto que las palabras sueltas.

Paso 3:

In [3]:
import pandas as pd
import numpy as np
import re
import unicodedata
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ==============================================================================
# RESPALDO DE CARGA Y PREPROCESAMIENTO (PASOS 1 Y 2)
# ==============================================================================
if 'df' not in globals():
    url = "https://raw.githubusercontent.com/eugeniomorocho/ArtificialIntelligence/main/Unit%205.%20Machine%20Learning%20as%20an%20AI%20Component/5.3%20Unsupervised%20learning/Clase_2/Recursos/catalogo_productos.csv"
    df = pd.read_csv(url)

cols_texto = df.select_dtypes(include=['object', 'string']).columns
col_desc = max(cols_texto, key=lambda c: df[c].astype(str).str.len().mean())

# Detectar columna de identificador si existe (ej. 'id', 'producto_id', 'codigo')
cols_id = [c for c in df.columns if 'id' in c.lower() or 'cod' in c.lower()]
col_id = cols_id[0] if cols_id else None

def quitar_acentos(texto):
    texto_norm = unicodedata.normalize('NFKD', texto)
    return "".join([c for c in texto_norm if not unicodedata.combining(c)])

def limpiar_texto(texto):
    texto = str(texto).lower()
    texto = quitar_acentos(texto)
    texto = re.sub(r'[^a-z0-9\s]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

stopwords_personalizadas = [
    'el', 'la', 'los', 'las', 'un', 'una', 'unos', 'unas',
    'de', 'del', 'al', 'a', 'en', 'por', 'para',
    'y', 'e', 'o', 'u', 'que', 'se', 'su', 'sus',
    'es', 'son', 'como', 'este', 'esta', 'estos', 'estas',
    'lo', 'le', 'les', 'me', 'te', 'nos'
]

# ==============================================================================
# 1. DETECCIÓN Y REGISTRO DE TEXTOS VACÍOS (CRUDOS Y POST-LIMPIEZA)
# ==============================================================================
nulos_crudos = df[col_desc].isna() | (df[col_desc].astype(str).str.strip() == "")
df["texto_limpio"] = df[col_desc].fillna("").apply(limpiar_texto)

# Vectorización TF-IDF (unigramas + bigramas con limpieza del Paso 2)
vectorizador = TfidfVectorizer(ngram_range=(1, 2), stop_words=stopwords_personalizadas)
X_tfidf = vectorizador.fit_transform(df["texto_limpio"])

# Calcular norma L2 de cada fila para detectar vectores nulos (ej. textos que solo tenían stopwords)
normas_filas = np.asarray(X_tfidf.power(2).sum(axis=1)).ravel()
vectores_nulos = np.isclose(normas_filas, 0.0, atol=1e-12)

print("=== 1. REGISTRO DE TEXTOS VACÍOS Y VECTORES NULOS ===")
print(f"Total de productos en catálogo: {len(df)}")
print(f"Textos vacíos o NaN en columna original ('{col_desc}'): {nulos_crudos.sum()}")
print(f"Vectores TF-IDF nulos tras limpieza y stopwords: {vectores_nulos.sum()}")

if vectores_nulos.sum() > 0:
    indices_vacios = df.index[vectores_nulos].tolist()
    print("Índices con vector nulo:", indices_vacios)
    print(df.loc[indices_vacios, [c for c in df.columns if c != "texto_limpio"]])

# ==============================================================================
# 2. MATRIZ DE SIMILITUD COSENO Y COMPROBACIÓN DE LA DIAGONAL
# ==============================================================================
sim_matrix = cosine_similarity(X_tfidf)
diagonal = np.diag(sim_matrix)

mascara_no_nulos = ~vectores_nulos
diag_no_nulos_ok = np.allclose(diagonal[mascara_no_nulos], 1.0, atol=1e-6)
diag_nulos_ok = np.allclose(diagonal[vectores_nulos], 0.0, atol=1e-6) if vectores_nulos.any() else True

print("\n=== 2. VERIFICACIÓN NUMÉRICA DE LA DIAGONAL ===")
print(f"Dimensión de la matriz de similitud: {sim_matrix.shape}")
print(f"Diagonal cercana a 1.0 (atol=1e-6) en {mascara_no_nulos.sum()} vectores no nulos: {diag_no_nulos_ok}")
print(f"Mínimo y máximo en diagonal (no nulos): {diagonal[mascara_no_nulos].min():.8f} / {diagonal[mascara_no_nulos].max():.8f}")
if vectores_nulos.any():
    print(f"Diagonal cercana a 0.0 en vectores nulos: {diag_nulos_ok}")

# ==============================================================================
# 3. RECUPERACIÓN DE TOP 3 SIMILARES (EXCLUYENDO POR ÍNDICE / ID)
# ==============================================================================
def buscar_similares(idx_consulta, top_k=3):
    """
    Recupera los top_k productos más similares excluyendo el producto consulta
    mediante su índice (y su ID si existe), evitando errores por empates en 1.0.
    """
    puntajes = sim_matrix[idx_consulta]
    id_consulta = df.loc[idx_consulta, col_id] if col_id else None

    # Filtrar explícitamente por índice (y por ID si está disponible)
    indices_candidatos = [
        i for i in range(len(df))
        if i != idx_consulta and (col_id is None or df.loc[i, col_id] != id_consulta)
    ]

    # Ordenar candidatos de mayor a menor similitud coseno
    top_indices = sorted(indices_candidatos, key=lambda i: puntajes[i], reverse=True)[:top_k]

    info_id = f" | {col_id}: {id_consulta}" if col_id else ""
    print(f"\n{'='*80}")
    print(f"CONSULTA [Índice {idx_consulta}{info_id}]:")
    print(f"  Descripción: {df.loc[idx_consulta, col_desc]}")
    print(f"{'-'*80}")
    print(f"TOP {top_k} PRODUCTOS MÁS SIMILARES:")

    for rank, idx_sim in enumerate(top_indices, start=1):
        sim_id = f" | {col_id}: {df.loc[idx_sim, col_id]}" if col_id else ""
        print(f"  {rank}. [Índice {idx_sim}{sim_id}] (Similitud Coseno: {puntajes[idx_sim]:.4f})")
        print(f"     {df.loc[idx_sim, col_desc]}")

# Ejecutar para al menos 3 productos distintos del catálogo (evitando índices nulos si los hay)
indices_validos = df.index[mascara_no_nulos].tolist()
consultas_ejemplo = [indices_validos[0], indices_validos[len(indices_validos)//3], indices_validos[-1]]

for idx_q in consultas_ejemplo:
    buscar_similares(idx_q, top_k=3)

=== 1. REGISTRO DE TEXTOS VACÍOS Y VECTORES NULOS ===
Total de productos en catálogo: 120
Textos vacíos o NaN en columna original ('descripcion'): 0
Vectores TF-IDF nulos tras limpieza y stopwords: 0

=== 2. VERIFICACIÓN NUMÉRICA DE LA DIAGONAL ===
Dimensión de la matriz de similitud: (120, 120)
Diagonal cercana a 1.0 (atol=1e-6) en 120 vectores no nulos: True
Mínimo y máximo en diagonal (no nulos): 1.00000000 / 1.00000000

CONSULTA [Índice 0 | producto_id: P001]:
  Descripción: telefono celular smartphone pantalla bateria camara compacto
--------------------------------------------------------------------------------
TOP 3 PRODUCTOS MÁS SIMILARES:
  1. [Índice 30 | producto_id: P031] (Similitud Coseno: 1.0000)
     telefono celular smartphone pantalla bateria camara compacto
  2. [Índice 60 | producto_id: P061] (Similitud Coseno: 1.0000)
     telefono celular smartphone pantalla bateria camara compacto
  3. [Índice 90 | producto_id: P091] (Similitud Coseno: 1.0000)
     telefono celul

En la verificación del catálogo se comprobó que existen 0 descripciones vacías y 0 vectores nulos tras el preprocesamiento, y que todos los valores de la diagonal principal de la matriz de similitud coseno son iguales a 1.0 dentro de la tolerancia numérica establecida. En un escenario real donde sí aparecieran descripciones vacías o que quedaran en blanco tras quitar palabras vacías, su vector TF-IDF tendría norma cero y devolvería similitud 0.0 contra todo el catálogo; para manejarlo se debería concatenar el nombre y la categoría del producto como respaldo antes de vectorizar, o marcar esos registros para recomendarles artículos populares de su misma categoría.

También se comprobó en la práctica la importancia de excluir el propio producto usando su índice y su `producto_id` en lugar de borrar simplemente la primera posición: como en este catálogo existen varios productos distintos con exactamente la misma descripción (por ejemplo, `P001`, `P031`, `P061` y `P091` tienen similitud coseno de 1.0000 entre sí), un corte por posición podría dejar fuera un vecino válido o devolver el mismo artículo consultado. Finalmente, calcular y guardar la matriz completa de 120 × 120 funciona sin problemas para este ejercicio, pero en una tienda con cientos de miles o millones de productos sería inviable en memoria RAM; en ese caso convendría mantener la matriz dispersa indexada y calcular la similitud únicamente contra el vector de la consulta actual o usar búsqueda aproximada de vecinos más cercanos.

Paso 4:

In [4]:
def explicar_coincidencias(idx_consulta, top_k=3):
    puntajes = sim_matrix[idx_consulta]
    candidatos = [i for i in range(len(df)) if i != idx_consulta]
    top_indices = sorted(candidatos, key=lambda i: puntajes[i], reverse=True)[:top_k]

    nombres_vocab = vectorizador.get_feature_names_out()
    vec_q = X_tfidf[idx_consulta].toarray().ravel()

    print(f"CONSULTA [{idx_consulta}]: {df.loc[idx_consulta, col_desc]}\n")
    for rank, idx_sim in enumerate(top_indices, 1):
        vec_s = X_tfidf[idx_sim].toarray().ravel()
        # Términos presentes en ambos documentos ordenados por aporte al producto punto
        contribucion = vec_q * vec_s
        idx_comunes = np.where(contribucion > 0)[0]
        idx_comunes = idx_comunes[np.argsort(contribucion[idx_comunes])[::-1]]
        terminos_comunes = [nombres_vocab[i] for i in idx_comunes[:8]]

        print(f"  {rank}. [{idx_sim}] Similitud: {puntajes[idx_sim]:.4f}")
        print(f"     Descripción: {df.loc[idx_sim, col_desc]}")
        print(f"     Términos clave compartidos: {terminos_comunes}\n")

# Usa los mismos dos índices que analizaste en el Paso 3
explicar_coincidencias(consultas_ejemplo[0], top_k=3)
print("=" * 80)
explicar_coincidencias(consultas_ejemplo[1], top_k=3)

CONSULTA [0]: telefono celular smartphone pantalla bateria camara compacto

  1. [30] Similitud: 1.0000
     Descripción: telefono celular smartphone pantalla bateria camara compacto
     Términos clave compartidos: ['camara compacto', 'telefono', 'smartphone pantalla', 'smartphone', 'telefono celular', 'pantalla bateria', 'celular', 'camara']

  2. [60] Similitud: 1.0000
     Descripción: telefono celular smartphone pantalla bateria camara compacto
     Términos clave compartidos: ['camara compacto', 'telefono', 'smartphone pantalla', 'smartphone', 'telefono celular', 'pantalla bateria', 'celular', 'camara']

  3. [90] Similitud: 1.0000
     Descripción: telefono celular smartphone pantalla bateria camara compacto
     Términos clave compartidos: ['camara compacto', 'telefono', 'smartphone pantalla', 'smartphone', 'telefono celular', 'pantalla bateria', 'celular', 'camara']

CONSULTA [40]: zapatillas calzado correr deporte ligero compacto

  1. [10] Similitud: 1.0000
     Descripción:

Al revisar las recomendaciones generadas para las dos consultas del Paso 3, se observa que el sistema recupera productos coherentes dentro de cada categoría, aunque con matices importantes desde el punto de vista del negocio:

1. **Consulta 1 (`P001` - `telefono celular smartphone pantalla bateria camara compacto`):** Las tres recomendaciones (`P031`, `P061` y `P091`) obtienen una similitud de 1.0000 porque tienen exactamente la misma descripción palabra por palabra. En términos comerciales, sí son sustitutos directos (todos pertenecen a la categoría de telefonía con las mismas características generales), pero el sistema no logra distinguir si son modelos distintos, gamas diferentes o simplemente publicaciones duplicadas del mismo teléfono.
2. **Consulta 2 (`P041` - `zapatillas calzado correr deporte ligero compacto`):** Ocurre el mismo fenómeno con `P011`, `P071` y `P101` (similitud de 1.0000). Las recomendaciones tienen sentido de negocio porque sugieren calzado deportivo para correr, evitando mezclar accesorios de otras categorías, pero nuevamente la coincidencia se debe a que las descripciones son plantillas idénticas y no a una comparación real de talla, marca o tipo de pisada.

**Decisión de validación antes de probar con usuarios reales:** Como evaluador, **no aprobaría todavía el paso directo a una prueba controlada con usuarios reales** sin antes hacer dos ajustes previos. Aunque el modelo no comete errores graves de categoría, en un catálogo real recomendar productos con descripciones idénticas y sin diferenciación de precio, marca o especificaciones técnicas genera una experiencia repetitiva para el cliente. Antes de exponerlo a usuarios reales, convendría enriquecer el texto combinando el `nombre` y la `categoria` con la `descripcion`, y aplicar reglas de negocio básicas (como evitar mostrar duplicados exactos o filtrar por disponibilidad y rango de precio).

Paso 5:

El principal problema de usar TF-IDF en un catálogo real es que depende por completo de que las palabras coincidan exactamente letra por letra. Como el algoritmo crea una dimensión distinta para cada término único del vocabulario, no tiene manera de saber que dos palabras diferentes apuntan al mismo objeto. Por ejemplo, si un vendedor publica un *"celular con cámara de 50MP"* y otro sube un artículo prácticamente idéntico pero lo describe como *"smartphone con lente de 50MP"*, para TF-IDF las palabras "celular" y "smartphone" son completamente independientes y no suman nada a la similitud. Esto ocurre con mucha frecuencia en tiendas en línea donde cada vendedor redacta a su manera, usa regionalismos o mezcla términos en inglés y español, como pasa entre "audífonos" y "auriculares", "laptop" y "portátil", o "mouse inalámbrico" y "ratón sin cable". Cuando sucede esto, el puntaje de similitud entre ambos vectores cae casi a cero y el sistema deja fuera del top de recomendaciones a productos que en realidad son los mejores sustitutos para el cliente.

Para superar esa limitación se podría dar el salto a los embeddings densos como Word2Vec, que revisamos en el subtema 3. A diferencia de TF-IDF, Word2Vec no se fija en la coincidencia exacta de caracteres, sino que aprende el significado de las palabras a partir del contexto en el que suelen aparecer dentro de textos grandes. Como "celular" y "smartphone" aparecen rodeados de palabras muy parecidas (como batería, pantalla, memoria o cargador), el modelo les asigna vectores muy cercanos entre sí dentro del espacio geométrico. Al representar cada producto promediando los vectores de las palabras que componen su descripción, el sistema logra detectar que ambos artículos son equivalentes aunque no compartan ni una sola palabra exacta.

Sin embargo, llevar Word2Vec a la práctica trae varios costos adicionales que hay que evaluar antes de cambiar el sistema. En primer lugar, no se puede entrenar un buen modelo de Word2Vec usando únicamente las descripciones de este catálogo porque es muy pequeño y no tendría suficiente contexto; habría que descargar e integrar un modelo preentrenado en español, lo que implica cargar archivos pesados en memoria RAM y aumentar los recursos necesarios para ejecutar el servicio. En segundo lugar, se pierde la eficiencia de las matrices dispersas de TF-IDF (donde solo se almacenan los valores distintos de cero) y se pasa a trabajar con vectores densos que ocupan más memoria y exigen mayor capacidad de cálculo al buscar vecinos cercanos. Además, Word2Vec suele tener dificultades con palabras fuera de su vocabulario o con códigos técnicos muy específicos (como referencias de modelos, tallas o puertos tipo "USB-C"), que es justo donde la coincidencia exacta de TF-IDF funciona mejor, y también hace más difícil auditar o explicar por qué el sistema recomendó un producto frente a otro.

Finalmente, si analizamos qué pasa cuando se publica un producto nuevo que todavía no tiene visitas, clics ni compras registradas, la gran ventaja del enfoque basado en contenido que construimos es que sí puede generar recomendaciones desde el primer día. A diferencia de los sistemas de filtrado colaborativo —que necesitan un historial previo de interacciones de los usuarios y quedan ciegos ante artículos recién creados—, nuestro modelo depende únicamente del texto descriptivo del producto. En la práctica, apenas un vendedor publica el artículo, solo hay que pasar su descripción por la misma función de limpieza, transformarla con el vectorizador ya entrenado (`vectorizador.transform`) y calcular su similitud coseno contra la matriz del catálogo existente para obtener al instante productos parecidos. Eso sí, para que esto funcione bien desde el arranque, es indispensable que el vendedor redacte una descripción con suficiente detalle (si la deja vacía o demasiado genérica no habrá señal para compararlo), y conviene actualizar el vectorizador periódicamente para que vaya incorporando términos o marcas nuevas que no existían cuando se ajustó por primera vez.